# Demonstrator 1: Q-learning w dyskretnym korytarzu 1D

**To jest gotowy przykład do uruchamiania i eksperymentowania — nie zadanie do rozwiązania.**
Jego celem jest pokazanie, jak działa tablicowy Q-learning, zanim napiszesz własny kod
w ćwiczeniu z `path_follower` (zob. `README.md`, sekcje 3 i 6).

Agent porusza się po korytarzu złożonym z kilku pól i ma dojść do celu na prawym końcu:

```
[A][ ][ ][ ][ ][G]      pola 0 1 2 3 4 5, cel G = pole 5
```

| Element | Znaczenie |
|---|---|
| stan `s` | indeks pola, `0 … N-1` (jest już dyskretny, nie trzeba nic robić) |
| akcje `a` | `0` = w lewo, `1` = w prawo |
| nagroda `r` | `+10` za wejście na cel, `-1` za każdy inny krok |
| koniec epizodu | wejście na pole celu (stan terminalny) lub limit kroków |

Aktualizacja (README, sekcja 3.3):

$$Q(s,a) \leftarrow Q(s,a) + \alpha\,[\,r + \gamma \max_{a'} Q(s',a') - Q(s,a)\,]$$

W stanie **terminalnym** składnik $\gamma \max_{a'} Q(s',a')$ pomijamy.

**Jak korzystać:** uruchamiaj komórki po kolei. Parametry do zmiany są w komórce
„Parametry". Po zmianie uruchom ponownie wszystko od tej komórki w dół.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'matplotlib'

## 1. Parametry (tu eksperymentuj)

In [ ]:
# --- środowisko ---
N_CELLS = 6            # liczba pól korytarza [-]; cel to zawsze ostatnie pole (N_CELLS - 1)
RANDOM_START = True    # True: losowy start spośród pól innych niż cel, False: zawsze pole 0
MAX_STEPS = 100        # limit kroków w jednym epizodzie [kroki]

# --- nagrody ---
REWARD_GOAL = 10.0     # nagroda za wejście na cel [-]
REWARD_STEP = -1.0     # nagroda (kara) za każdy pozostały krok [-]

# --- Q-learning ---
N_EPISODES = 300       # liczba epizodów uczenia [-]
ALPHA = 0.5            # współczynnik uczenia, (0, 1] [-]
GAMMA = 0.9            # współczynnik dyskontowania, (0, 1] [-]

# --- strategia epsilon-zachłanna ---
EPSILON_START = 1.0    # epsilon w pierwszym epizodzie, [0, 1] [-]
EPSILON_END = 0.05     # epsilon w ostatnim epizodzie, [0, 1] [-]
EPSILON_MODE = "linear"  # "const" (stałe EPSILON_START), "linear" lub "exp" (redukcja)

SEED = 0               # ziarno generatora losowego (powtarzalność wyników)

## 2. Środowisko

Środowisko zna tylko zasady „fizyki" świata: jak zmienia się pozycja i jaką nagrodę
agent dostaje. **Nie zna** go algorytm uczący — widzi tylko `(stan, nagroda, koniec)`.
To właśnie znaczy, że Q-learning jest *bezmodelowy*.

In [ ]:
ACTIONS = ["lewo", "prawo"]          # nazwy akcji; indeks akcji = pozycja na liście
ACTION_DELTA = [-1, +1]              # zmiana pozycji dla każdej akcji [pola]
N_ACTIONS = len(ACTIONS)
GOAL = N_CELLS - 1                   # indeks pola celu


class CorridorEnv:
    """Korytarz 1D. Obserwacją jest indeks pola, na którym stoi agent."""

    def __init__(self, n_cells, random_start):
        self.n_cells = n_cells
        self.random_start = random_start
        self.position = 0            # aktualne pole agenta [indeks pola]

    def reset(self, rng):
        """Zaczyna nowy epizod i zwraca obserwację początkową."""
        if self.random_start:
            self.position = int(rng.integers(0, self.n_cells - 1))  # bez pola celu
        else:
            self.position = 0
        return self.position

    def step(self, action):
        """Wykonuje akcję. Zwraca (obserwacja, nagroda, czy_terminalny)."""
        if not 0 <= action < N_ACTIONS:
            raise ValueError(f"Niepoprawny numer akcji: {action}")
        # próba wyjścia poza korytarz zostawia agenta na skrajnym polu
        self.position = int(np.clip(self.position + ACTION_DELTA[action], 0, self.n_cells - 1))
        if self.position == GOAL:
            return self.position, REWARD_GOAL, True
        return self.position, REWARD_STEP, False


def draw_corridor(position, n_cells=None):
    """Zwraca tekstową wizualizację korytarza, np. `[ ][A][ ][ ][ ][G]`."""
    n_cells = n_cells or N_CELLS
    cells = []
    for i in range(n_cells):
        if i == position:
            cells.append("[A]")
        elif i == n_cells - 1:
            cells.append("[G]")
        else:
            cells.append("[ ]")
    return "".join(cells)


for p in range(N_CELLS):
    print(f"pole {p}: {draw_corridor(p)}")

## 3. Reprezentacja stanu

W korytarzu stan **jest** pozycją: `stan = pozycja_na_planszy`.
Funkcja `encode_state` jest więc tożsamością. W drugim demonstratorze (parkowanie 1D)
to właśnie ta funkcja będzie jedyną dużą różnicą — tam obserwacja jest ciągła i trzeba ją
zdyskretyzować. W ćwiczeniu odpowiada jej `discretize_state`.

In [ ]:
def encode_state(observation):
    """Zamienia obserwację na indeks wiersza tablicy Q. Tu: bez zmian."""
    return int(observation)

## 4. Wybór akcji, aktualizacja Q, pętla uczenia

Poniższe funkcje są **identyczne** w obu demonstratorach.

In [ ]:
def choose_action(q_table, state, epsilon, rng):
    """Strategia epsilon-zachłanna.

    Z prawdopodobieństwem `epsilon` zwraca akcję losową (eksploracja), w przeciwnym razie
    akcję o największej wartości Q (eksploatacja). Remisy (np. na początku, gdy cała tablica
    to zera) są rozstrzygane losowo — inaczej agent zawsze brałby akcję 0.
    """
    n_actions = q_table.shape[1]
    if rng.random() < epsilon:
        return int(rng.integers(n_actions))
    best = np.flatnonzero(q_table[state] == q_table[state].max())
    return int(rng.choice(best))


def update_q_table(q_table, state, action, reward, next_state, terminal, alpha, gamma):
    """Jedna aktualizacja Q-learningu dla przejścia (s, a, r, s')."""
    if terminal:
        target = reward                                           # brak następnego kroku
    else:
        target = reward + gamma * np.max(q_table[next_state])     # r + gamma * max Q(s', a')
    td_error = target - q_table[state, action]                    # błąd różnicy czasowej
    q_table[state, action] += alpha * td_error


def epsilon_for_episode(episode, n_episodes, eps_start, eps_end, mode):
    """Wartość epsilon w danym epizodzie (numerowanym od 0)."""
    if mode == "const":
        return eps_start
    progress = episode / max(1, n_episodes - 1)                   # 0 … 1
    if mode == "linear":
        return eps_start + (eps_end - eps_start) * progress
    if mode == "exp":
        return eps_start * (eps_end / eps_start) ** progress
    raise ValueError(f"Nieznany tryb epsilon: {mode}")


def train(env, n_states, n_actions, n_episodes, alpha, gamma,
          eps_start, eps_end, eps_mode, max_steps, seed):
    """Uczy agenta. Zwraca tablicę Q i historię uczenia (po jednej wartości na epizod)."""
    rng = np.random.default_rng(seed)
    q_table = np.zeros((n_states, n_actions))
    history = {"return": [], "steps": [], "success": [], "epsilon": []}

    for episode in range(n_episodes):
        epsilon = epsilon_for_episode(episode, n_episodes, eps_start, eps_end, eps_mode)
        state = encode_state(env.reset(rng))
        total_reward, terminal, steps = 0.0, False, 0

        while not terminal and steps < max_steps:
            action = choose_action(q_table, state, epsilon, rng)
            observation, reward, terminal = env.step(action)
            next_state = encode_state(observation)
            # Aktualizacja dopiero PO poznaniu skutków akcji (r i s').
            # Limit kroków NIE jest stanem terminalnym, więc nadal liczymy max Q(s', .).
            update_q_table(q_table, state, action, reward, next_state, terminal, alpha, gamma)
            state = next_state
            total_reward += reward
            steps += 1

        history["return"].append(total_reward)
        history["steps"].append(steps)
        history["success"].append(terminal)
        history["epsilon"].append(epsilon)
    return q_table, history

## 5. Początkowa tablica Q

Wiersze to stany (pola), kolumny to akcje. Na początku agent nic nie wie, więc same zera.

In [ ]:
env = CorridorEnv(N_CELLS, RANDOM_START)


def print_q_table(q_table):
    print("pole |  Q(lewo)  Q(prawo) | najlepsza akcja")
    for s, row in enumerate(q_table):
        best = "— (cel)" if s == GOAL else ACTIONS[int(np.argmax(row))]
        print(f"{s:4d} | {row[0]:8.2f} {row[1]:9.2f} | {best}")


print_q_table(np.zeros((N_CELLS, N_ACTIONS)))

## 6. Trening

In [ ]:
q_table, history = train(env, N_CELLS, N_ACTIONS, N_EPISODES, ALPHA, GAMMA,
                         EPSILON_START, EPSILON_END, EPSILON_MODE, MAX_STEPS, SEED)
print(f"Trening zakończony: {N_EPISODES} epizodów, "
      f"skuteczność w ostatnich 50 epizodach (z eksploracją): "
      f"{np.mean(history['success'][-50:]):.0%}")

### Co zobaczyć na wykresach
* **Suma nagród** powinna rosnąć i się stabilizować. Maksimum zależy od startu:
  z pola 0 optymalna droga to 4 kroki po `-1` i krok końcowy `+10`, czyli `+6`.
* **Liczba kroków** powinna spadać do najkrótszej drogi (od 1 do `N_CELLS-1` kroków).
* **Epsilon** maleje, więc wraz z nim ubywa losowych ruchów. Pierwsze epizody są
  „chaotyczne", bo agent głównie eksploruje.
* Wygładzamy wykres średnią kroczącą, bo pojedyncze epizody są bardzo szumiące.

In [ ]:
def moving_average(values, window=20):
    values = np.asarray(values, dtype=float)
    if len(values) < window:
        return values
    return np.convolve(values, np.ones(window) / window, mode="valid")


fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history["return"], alpha=0.3, label="pojedynczy epizod")
ma = moving_average(history["return"])
axes[0].plot(np.arange(len(history["return"]) - len(ma), len(history["return"])), ma, label="średnia krocząca (20)")
axes[0].set(xlabel="numer epizodu [-]", ylabel="suma nagród [-]", title="Suma nagród w epizodzie")
axes[0].legend()

axes[1].plot(history["steps"], alpha=0.3, label="pojedynczy epizod")
ma = moving_average(history["steps"])
axes[1].plot(np.arange(len(history["steps"]) - len(ma), len(history["steps"])), ma, label="średnia krocząca (20)")
axes[1].set(xlabel="numer epizodu [-]", ylabel="liczba kroków [kroki]", title="Długość epizodu")
axes[1].legend()

axes[2].plot(history["epsilon"], color="tab:red", label="epsilon")
axes[2].set(xlabel="numer epizodu [-]", ylabel="epsilon [-]", title="Eksploracja")
axes[2].legend()
plt.tight_layout()
plt.show()

## 7. Końcowa tablica Q i wyuczona polityka

Polityka to po prostu $\pi(s) = \arg\max_a Q(s,a)$. Oczekujemy ruchu w **prawo** z każdego
pola poza celem. Wartości Q w stanie celu pozostają zerami — z celu nie wykonuje się już
żadnej akcji (stan terminalny).

Sprawdź, czy wartości Q rosną w stronę celu: im bliżej celu, tym wyższe `Q(prawo)`.

In [ ]:
print_q_table(q_table)

policy = np.argmax(q_table, axis=1)
print("\nPolityka:")
print("".join(f"[{'<' if policy[s] == 0 else '>'}]" if s != GOAL else "[G]" for s in range(N_CELLS)))

fig, ax = plt.subplots(figsize=(6, 3.5))
im = ax.imshow(q_table.T, cmap="viridis", aspect="auto")
ax.set_xticks(range(N_CELLS))
ax.set_yticks(range(N_ACTIONS))
ax.set_yticklabels(ACTIONS)
ax.set(xlabel="stan: numer pola [-]", ylabel="akcja", title="Tablica Q")
fig.colorbar(im, label="Q [-]")
plt.show()

## 8. Demonstracja bez eksploracji (epsilon = 0)

Agent korzysta wyłącznie z wyuczonej tablicy. Wypisujemy każdy krok.
Możesz zmienić `DEMO_START` (pole 0 … `N_CELLS-2`).

In [ ]:
DEMO_START = 0         # pole startowe demonstracji [indeks pola]

demo_env = CorridorEnv(N_CELLS, random_start=False)
demo_env.reset(np.random.default_rng(SEED))
demo_env.position = DEMO_START
state = encode_state(demo_env.position)
rng_demo = np.random.default_rng(SEED)

print(f"start    {draw_corridor(demo_env.position)}")
for t in range(1, MAX_STEPS + 1):
    action = choose_action(q_table, state, 0.0, rng_demo)
    observation, reward, terminal = demo_env.step(action)
    state = encode_state(observation)
    print(f"krok {t:2d}  {draw_corridor(observation)}  akcja={ACTIONS[action]:5s} r={reward:+.0f}")
    if terminal:
        print("Cel osiągnięty.")
        break
else:
    print("Limit kroków — cel nieosiągnięty.")

## 9. Zapis tablicy Q do pliku tekstowego

W ćwiczeniu będziesz zapisywać i wczytywać Q-tablicę z pliku. Przykładowy format z `README.md`
(sekcja 6.4): jeden wiersz na stan — najpierw liczba cech stanu `N`, potem `N` indeksów,
potem wartości Q dla kolejnych akcji. Tu stan ma jedną cechę, więc wiersz wygląda tak:
`1 <pole> <Q_lewo> <Q_prawo>`. Niepoprawne wiersze są odrzucane w całości z ostrzeżeniem,
a brak pliku oznacza pustą tablicę.

In [ ]:
import math
import os

Q_FILE = "q_table_korytarz.txt"   # plik wyjściowy (ścieżka względna do katalogu notebooka)


def save_q_table(q_table, filename):
    with open(filename, "w") as f:
        for s, row in enumerate(q_table):
            f.write("1 " + str(s) + " " + " ".join(f"{v:.6f}" for v in row) + "\n")


def load_q_table(filename, n_states, n_actions):
    q = np.zeros((n_states, n_actions))
    if not os.path.exists(filename):
        return q
    with open(filename) as f:
        for line_no, line in enumerate(f, start=1):
            parts = line.split()
            if not parts:
                continue
            try:
                n_features = int(parts[0])
                if n_features != 1 or len(parts) != 1 + n_features + n_actions:
                    raise ValueError("zła liczba pól")
                s = int(parts[1])
                values = [float(x) for x in parts[2:]]
                if not 0 <= s < n_states or not all(math.isfinite(v) for v in values):
                    raise ValueError("stan poza zakresem lub wartość niefinitywna")
            except ValueError as err:
                print(f"UWAGA: pominięto wiersz {line_no}: {err}")
                continue
            q[s] = values
    return q


save_q_table(q_table, Q_FILE)
loaded = load_q_table(Q_FILE, N_CELLS, N_ACTIONS)
print("Tablica po zapisie i wczytaniu jest taka sama:", np.allclose(q_table, loaded, atol=1e-6))
with open(Q_FILE) as f:
    print(f.read())

## 10. Co spróbować zmienić

1. `ALPHA` = 0.05, 0.5, 1.0 — jak zmienia się tempo uczenia i szum na wykresie?
2. `GAMMA` = 0.0, 0.5, 0.99 — dla `GAMMA = 0` agent patrzy tylko na natychmiastową nagrodę.
   Czy nadal znajdzie drogę do celu? (Wskazówka: nagroda jest tylko na ostatnim kroku.)
3. `EPSILON_MODE = "const"` i `EPSILON_START = 0.0` — agent bez eksploracji. Co się stanie,
   gdy `REWARD_STEP = 0.0`? Dlaczego kara za krok pomaga wybrać najkrótszą drogę?
4. `EPSILON_START` = `EPSILON_END` = 0.5 — wysoki, stały epsilon. Porównaj wykres
   nagród i politykę (demonstracja jest z epsilon = 0, ale uczenie było zaszumione).
5. `RANDOM_START = False` — agent zawsze startuje z pola 0 i uczy się tylko tej drogi.
   Czy wiersze tablicy Q dla pól bliższych celu zostały w ogóle zaktualizowane?
6. `N_CELLS` = 10 i 20 — ile epizodów potrzeba, żeby polityka była poprawna?
   Tablica rośnie liniowo, ale nagroda musi „dotrzeć" od celu do początku korytarza.